# 第10章: 事前学習済み言語モデル（GPT型）

本章では、GPT型（Transformerのデコーダ型）の事前学習済みモデルを利用して、言語生成、評判分析器（ポジネガ分類器）の構築、ファインチューニング、強化学習などに取り組む。

## 90. 次単語予測

“The movie was full of"に続くトークン（トークン列ではなく一つのトークンであることに注意せよ）として適切なもの上位10個と、その確率（尤度）を求めよ。ただし、言語モデルへのプロンプトがどのようなトークン列に変換されたか、確認せよ。

In [ ]:
# transformersライブラリをインストールします。
!pip install transformers

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

# 事前学習済みGPT-2モデルとトークナイザーをロード
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

# 入力プロンプト
text = "The movie was full of"

# プロンプトをトークン化し、その結果を表示
input_ids = tokenizer.encode(text, return_tensors='pt')
print(f"入力テキスト: '{text}'")
print(f"トークン化されたID: {input_ids}")
print(f"トークン列: {[tokenizer.decode(token_id) for token_id in input_ids[0]]}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

入力テキスト: 'The movie was full of'
トークン化されたID: tensor([[ 464, 3807,  373, 1336,  286]])
トークン列: ['The', ' movie', ' was', ' full', ' of']


上記の出力で、`input_ids`が言語モデルへのプロンプトが変換されたトークン列のIDです。`トークン列`はそれぞれのIDに対応するトークンです。

次に、このトークン列の次に来る単語を予測し、上位10個のトークンとその確率を計算します。

In [ ]:
# モデルにプロンプトを与えて次トークンの予測を取得
with torch.no_grad():
    outputs = model(input_ids)
    predictions = outputs.logits

# 最後のトークンの次の予測（ロジット）を取得
next_token_logits = predictions[0, -1, :]

# 確率に変換するためにsoftmaxを適用
probabilities = torch.softmax(next_token_logits, dim=-1)

# 確率の高い上位10個のトークンとその確率を取得
top_10_probabilities, top_10_indices = torch.topk(probabilities, 10)

print("\n次トークンの予測 (上位10個):")
for i, (prob, idx) in enumerate(zip(top_10_probabilities, top_10_indices)):
    token = tokenizer.decode(idx)
    print(f"{i+1}. トークン: '{token}', 確率: {prob.item():.4f}")



次トークンの予測 (上位10個):
1. トークン: ' jokes', 確率: 0.0219
2. トークン: ' great', 確率: 0.0186
3. トークン: ' laughs', 確率: 0.0115
4. トークン: ' bad', 確率: 0.0109
5. トークン: ' surprises', 確率: 0.0107
6. トークン: ' references', 確率: 0.0105
7. トークン: ' fun', 確率: 0.0100
8. トークン: ' humor', 確率: 0.0074
9. トークン: ' "', 確率: 0.0074
10. トークン: ' the', 確率: 0.0067


## 91. 続きのテキストの予測

“The movie was full of"に続くテキストを複数予測せよ。このとき、デコーディングの方法や温度パラメータ（temperature）を変えながら、予測される複数のテキストの変化を観察せよ。

In [ ]:
import torch

# 入力プロンプト
text = "The movie was full of"
input_ids = tokenizer.encode(text, return_tensors='pt')

print(f"入力プロンプト: '{text}'\n")

# 異なるtemperatureでテキストを生成
def generate_text_with_temperature(input_ids, model, tokenizer, temperature, num_sequences=3, max_length=50):
    print(f"--- temperature={temperature} の場合 ---")
    generated_ids = model.generate(
        input_ids,
        max_length=max_length,
        num_return_sequences=num_sequences,
        no_repeat_ngram_size=2,
        do_sample=True,
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id
    )

    for i, generated_id in enumerate(generated_ids):
        generated_text = tokenizer.decode(generated_id, skip_special_tokens=True)
        print(f"生成されたテキスト {i+1}: {generated_text}")
    print("\n")

# temperatureが低い場合 (より保守的で予測可能)
generate_text_with_temperature(input_ids, model, tokenizer, temperature=0.7)

# temperatureが中程度の場合
generate_text_with_temperature(input_ids, model, tokenizer, temperature=1.0)

# temperatureが高い場合 (より多様で創造的だが、一貫性が低くなる可能性あり)
generate_text_with_temperature(input_ids, model, tokenizer, temperature=1.5)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


入力プロンプト: 'The movie was full of'

--- temperature=0.7 の場合 ---
生成されたテキスト 1: The movie was full of hilarious moments in which the characters (played by actors and actresses), all of whom were in a state of shock, were completely oblivious to the fact that they were wearing a bikini.

Then the character of the guy who
生成されたテキスト 2: The movie was full of good jokes and good laughs. I had the time to watch some of the more serious comedies, but I can't say I liked it better since I don't think I've ever seen any of them. The fact that
生成されたテキスト 3: The movie was full of scenes from the '70s and '80s: the great, wonderful-looking cars of the 1950s, the cool-sounding V-8s of '60s. The cars were all over the place, so


--- temperature=1.0 の場合 ---
生成されたテキスト 1: The movie was full of jokes about black people being "wicked," or how blacks aren't black enough. "No!" he yelled. His character yelled, "It's not white people!"

"He gets it pretty hard."



生成されたテキスト 2: The movie was full of great perfor

## 92. 予測されたテキストの確率を計算

“The movie was full of"に続くテキストを予測し、生成された各単語の尤度を表示せよ（生成されるテキストが長いと出力が読みにくくなるので、適当な長さで生成を打ち切るとよい）。

## 93. パープレキシティ

適当な文を準備して、事前学習済み言語モデルでパープレキシティを測定せよ。例えば、

+ The movie was full of surprises
+ The movies were full of surprises
+ The movie were full of surprises
+ The movies was full of surprises

の4文に対して、パープレキシティを測定して観察せよ（最後の2つの文は故意に文法的な間違いを入れた）。

## 94. チャットテンプレート

"What do you call a sweet eaten after dinner?"という問いかけに対する応答を生成するため、チャットテンプレートを適用し、言語モデルに与えるべきプロンプトを作成せよ。また、そのプロンプトに対する応答を生成し、表示せよ。

## 95. マルチターンのチャット

問題94で生成された応答に対して、追加で"Please give me the plural form of the word with its spelling in reverse order."と問いかけたときの応答を生成・表示せよ。また、その時に言語モデルに与えるプロンプトを確認せよ。

## 96. プロンプトによる感情分析

事前学習済み言語モデルで感情分析を行いたい。テキストを含むプロンプトを事前学習済み言語モデルに与え、（ファインチューニングは行わずに）テキストのポジネガを予測するという戦略で、[SST-2](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip)の開発データにおける正解率を測定せよ。

## 97. 埋め込みに基づく感情分析

事前学習済み言語モデルでテキストをベクトルで表現（エンコード）し、そのベクトルにフィードフォワード層を通すことで極性ラベルを予測するモデルを学習せよ。

## 98. ファインチューニング

問題96のプロンプトに対して、正解の感情ラベルをテキストの応答として返すように事前学習済みモデルをファインチューニングせよ。

## 99. 選好チューニング

問題96のプロンプトに対して、正解の感情ラベルを含むテキストを望ましい応答、間違った感情ラベルを含むテキストを望ましくない応答として、事前学習済み言語モデルを選好チューニング (preference tuning) を実施せよ。選好チューニングのアルゴリズムとしては、近傍方策最適化 (PPO: Proximal Policy Optimization) や直接選好最適化 (DPO: Direct Preference Optimization) などが考えられる。
